# Physique des matériaux – LMAPR1492 (Partie Prof. Rignanese)
## Notebook d'analyse du matériau : **[AlGaN2]**

### Membres du groupe
- Antoine Julianne
- Bohy Maria-Olivia 
- Dumont Louisa
- Salée de Béthune Apolline

### Matériau (Materials Project)
- Material ID : `mp-1008556`
- Formule : AlGaN₂

---
## Objectif du notebook
Ce notebook couvre les points demandés :
1. Télécharger la **structure cristalline**
2. Déterminer : réseau direct & réciproque, **type de maille**, **système cristallin**, **groupe ponctuel**
3. Montrer l'effet de **3 éléments de symétrie** (≠ identité), chacun sur **un atome différent**
4. Visualiser la **zone de Brillouin**
5. Calculer les **3 premiers pics** du diffractogramme (Cu Kα, λ = 1.54060 Å) et indices **hkl**
6. Télécharger et analyser la **structure de bandes électroniques** : gap, directions de dispersion max/min, masses effectives
7. Télécharger et analyser la **structure de bandes de phonons** : vitesses du son (3 branches, 3 directions); représenter les trois densités d'états de phonons correspondantes
8. Télécharger la **DOS phonons** et fitter **ΘD** (Debye) et **ΘE** (Einstein) + comparer Cv et DOS


# Importations des bibliothèques nécessaires


In [1]:
import numpy as np
import matplotlib.pyplot as plt

from pymatgen.ext.matproj import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.analysis.diffraction.xrd import XRDCalculator

# Band structure / phonons plotters (selon disponibilité MP)
from pymatgen.electronic_structure.plotter import BSPlotter
from pymatgen.phonon.plotter import PhononBSPlotter

from scipy.optimize import curve_fit
from scipy.constants import hbar, k as k_B

API_KEY = "METTRE_VOTRE_CLE_ICI"
MP_ID = "mp-1008556"  


## 0.1 Sanity check
- Version de pymatgen
- Connexion API


In [3]:
from pymatgen.core import __version__
print("pymatgen version:", __version__)


pymatgen version: 2025.10.7


# 1. Cristallographie géométrique — Structure du matériau


## 1.1 Télécharger la structure du matériau attribué
- Récupérer l'objet `Structure`
- Afficher une info de base (formule, nb d'atomes, paramètres de maille)


In [8]:
API_KEY = 'vKJsFu0jdhLy7CJj5Mwar6S68kxgMc3n'
with MPRester(API_KEY) as mpr:
    structure = mpr.get_structure_by_material_id(MP_ID)

print("Formule:", structure.composition.reduced_formula)
print("Nombre d'atomes dans la maille fournie:", len(structure))
print("Paramètres de maille (a,b,c) et (alpha,beta,gamma):", structure.lattice.abc, structure.lattice.angles)
structure


Formule: AlGaN2
Nombre d'atomes dans la maille fournie: 4
Paramètres de maille (a,b,c) et (alpha,beta,gamma): (3.140877048687329, 3.140877048687329, 4.42383794) (90.0, 90.0, 90.0)


Structure Summary
Lattice
    abc : 3.140877048687329 3.140877048687329 4.42383794
 angles : 90.0 90.0 90.0
 volume : 43.64164186160555
      A : np.float64(-2.22093546) np.float64(-2.22093546) np.float64(-0.0)
      B : np.float64(-2.22093546) np.float64(2.22093546) np.float64(0.0)
      C : np.float64(-0.0) np.float64(-0.0) np.float64(-4.42383794)
    pbc : True True True
PeriodicSite: Al (-2.221, 0.0, -2.212) [0.5, 0.5, 0.5]
PeriodicSite: Ga (0.0, 0.0, 0.0) [0.0, 0.0, -0.0]
PeriodicSite: N (-1.11, 1.11, -1.158) [0.0, 0.5, 0.2617]
PeriodicSite: N (-1.11, -1.11, -3.266) [0.5, 0.0, 0.7383]

## 1.2 Réseau direct : vecteurs de base, volume, type de maille
À produire :
- vecteurs **a, b, c** (cartésiens) : **a** = a**x**, **b** = a**x**, **c** = c**x** (a = b)
- volume Ω : **a**.(**b**x**c**) = **b**.(**c**x**a**) = **c**.(**a**x**b**) = a²c = 43.64 Å³
- commentaire : maille primitive / conventionnelle / nombre d'atomes = 4


In [9]:
lattice = structure.lattice
a_vec, b_vec, c_vec = lattice.matrix
print("a =", a_vec)
print("b =", b_vec)
print("c =", c_vec)
print("Volume Ω =", lattice.volume)


a = [-2.22093546 -2.22093546 -0.        ]
b = [-2.22093546  2.22093546  0.        ]
c = [-0.         -0.         -4.42383794]
Volume Ω = 43.64164186160555


## 1.3 Système cristallin, type de maille, groupe d'espace, groupe ponctuel
À produire :
- système cristallin : tétragonal (a = b != c; alpha = beta = gamme = 90°)
- symbole & numéro du groupe d'espace : P -4 m 2, 115
- groupe ponctuel : -4 2 m


In [10]:
sga = SpacegroupAnalyzer(structure)
spg_symbol = sga.get_space_group_symbol()
spg_number = sga.get_space_group_number()
crys_system = sga.get_crystal_system()
point_group = sga.get_point_group_symbol()

print("Système cristallin:", crys_system)
print("Groupe d'espace:", spg_symbol, "(#", spg_number, ")")
print("Groupe ponctuel:", point_group)


Système cristallin: tetragonal
Groupe d'espace: P-4m2 (# 115 )
Groupe ponctuel: -42m


## 1.4 Réseau réciproque : vecteurs de base
À produire :
- vecteurs **b1, b2, b3** du réseau réciproque
- rappel : **b1** = 2π (**b**x**c**)/Ω = 2π/**a**; **b2** = 2π (**c**x**a**)/Ω = 2π/**a**;  **b3** = 2π (**a**x**b**)/Ω = 2π/**c**

Réseau réciproque est un tétragonal primitif : **b1 = b2 != b3**


In [11]:
rec = lattice.reciprocal_lattice
b1, b2, b3 = rec.matrix
print("b1 =", b1)
print("b2 =", b2)
print("b3 =", b3)


b1 = [-1.41453577 -1.41453577 -0.        ]
b2 = [-1.41453577  1.41453577 -0.        ]
b3 = [-0.          0.         -1.42030187]


## 1.5 Zone de Brillouin
À produire :
- visualisation de la 1ère zone de Brillouin
- liste des points de haute symétrie utilisés dans les bandes (Γ, X, L, K, ...)
> Selon les outils, on peut au minimum illustrer la BZ ou afficher le chemin k utilisé par MP.


In [ ]:
# Option simple: afficher la lattice réciproque et tenter la BZ
bz = lattice.get_brillouin_zone()
bz


## 1.6 Symétries : montrer l'effet de 3 éléments de symétrie (≠ identité)
Consignes :
- choisir **3 opérations de symétrie différentes**
- appliquer chacune à **un atome différent**
- illustrer avant/après (coordonnées + figure si possible)

À remplir :
- Opération 1 : [rotation / miroir / inversion / roto-inversion / ...]
- Opération 2 : ...
- Opération 3 : ...


In [ ]:
# Récupérer les opérations de symétrie
symm_ops = sga.get_symmetry_operations(cartesian=False)  # en coordonnées réduites
print("Nombre d'opérations de symétrie:", len(symm_ops))
symm_ops[:5]


In [ ]:
# Choisir 3 opérations NON identités (à adapter)
# Astuce: vérifier op.is_identity
non_id_ops = [op for op in symm_ops if not op.is_identity]
print("Non-identités:", len(non_id_ops))

# Choisir 3 ops (à ajuster si nécessaire)
op1, op2, op3 = non_id_ops[0], non_id_ops[1], non_id_ops[2]

# Choisir 3 atomes différents (indices)
i1, i2, i3 = 0, 1, 2  # <-- à adapter si <3 atomes ou si vous voulez d'autres atomes

def apply_op(frac_coords, op):
    return op.operate(frac_coords)  # retourne des coords réduites (peut sortir de [0,1[)

for (i, op, name) in [(i1, op1, "Opération 1"), (i2, op2, "Opération 2"), (i3, op3, "Opération 3")]:
    fc = structure[i].frac_coords
    fc2 = apply_op(fc, op)
    print(name)
    print("  Atome:", structure[i].specie, "index", i)
    print("  Avant (frac):", fc)
    print("  Après  (frac):", fc2)
    print("  Matrice rotation:
", op.rotation_matrix)
    print("  Vecteur translation:", op.translation_vector)
    print()


# 2. Radiocristallographie — Diffraction RX


## 2.1 Visualiser le diffractogramme (Cu Kα, λ = 1.54060 Å)
- générer le pattern
- tracer intensité vs 2θ


In [ ]:
xrd = XRDCalculator(wavelength=1.54060)
pattern = xrd.get_pattern(structure)

plt.figure()
plt.plot(pattern.x, pattern.y)
plt.xlabel("2θ (deg)")
plt.ylabel("Intensité (a.u.)")
plt.title(f"XRD (Cu Kα) – {structure.composition.reduced_formula} ({MP_ID})")
plt.show()


## 2.2 Déterminer les 3 premiers pics et indices (hkl)
Consigne :
- **3 premiers pics** (les plus petits 2θ, non nuls)
- indiquer les indices **hkl** associés
- commenter (extinctions, type de maille, etc. si pertinent)


In [ ]:
# Les pics sont déjà triés par 2θ croissant
for i in range(3):
    print(f"Pic {i+1}: 2θ = {pattern.x[i]:.3f} deg | Intensité = {pattern.y[i]:.2f} | hkls = {pattern.hkls[i]}")


### Tableau récapitulatif à compléter
| Pic | 2θ (deg) | d_hkl (Å) | (hkl) dominant | Commentaire |
|---|---:|---:|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |


In [ ]:
# Option: calculer d à partir de 2θ (Bragg) : 2 d sinθ = λ
lam = 1.54060
two_theta = np.array(pattern.x[:3])
theta = np.deg2rad(two_theta/2)
d = lam/(2*np.sin(theta))
d


# 3. Structure de bandes électroniques — Analyse


## 3.1 Télécharger la structure de bandes électroniques
- récupérer band structure depuis Materials Project
- tracer la figure


In [ ]:
with MPRester(API_KEY) as mpr:
    bs = mpr.get_bandstructure_by_material_id(MP_ID)

plotter = BSPlotter(bs)
plotter.get_plot()
plt.show()


## 3.2 Déterminer la bande interdite
- valeur du gap (eV)
- direct / indirect
- k-points associés (si disponible)


In [ ]:
gap_info = bs.get_band_gap()
gap_info


## 3.3 Direction de dispersion max/min (valence & conduction)
Consigne :
- trouver la **direction** dans laquelle la **dernière bande de valence** est :
  - la plus dispersive
  - la moins dispersive
- idem pour la **première bande de conduction**
- insérer une figure avec des flèches indiquant la dispersion

À faire :
1) Identifier VBmax & CBmin
2) Pour chaque segment du chemin k, estimer ΔE/Δk (ou ΔE) et comparer


In [ ]:
# PISTE: récupérer les données du BS pour analyser dispersion le long des branches.
# Selon le type d'objet (line_mode), bs.branches contient les segments et labels.
print("Type:", type(bs))
print("Nombre de branches:", len(bs.branches))
bs.branches[:3]


In [ ]:
# TODO: Implémenter une mesure de dispersion par branche:
# - Extraire E(k) de la bande VB (index) et CB (index)
# - Mesurer amplitude (max-min) par branche
# - Associer à labels de la branche (start_label -> end_label)
#
# Indice de bande: à déterminer à partir du remplissage (nb e-), ou en utilisant bs.get_vbm()/get_cbm()
vbm = bs.get_vbm()
cbm = bs.get_cbm()
print("VBM:", vbm['energy'], "eV | kpoint:", vbm['kpoint'].label, vbm['kpoint'].frac_coords)
print("CBM:", cbm['energy'], "eV | kpoint:", cbm['kpoint'].label, cbm['kpoint'].frac_coords)


### Figure à insérer (dispersion)
- Copier la figure des bandes
- Ajouter flèches sur les directions "plus" et "moins" dispersives
- Expliquer en 3–5 lignes le lien dispersion ↔ masse effective


## 3.4 Masse effective (approximation parabolique)
Consigne :
- calculer la masse effective au sommet de la dernière bande de valence (autour VBM)
- calculer la masse effective à la base de la première bande de conduction (autour CBM)
Hypothèse : dispersion parabolique E(k) ≈ E0 + α (k-k0)^2
Formule (1D le long d'une direction) :
m* = ħ² / (d²E/dk²)

À faire :
- choisir une direction / segment autour de k0
- sélectionner quelques points proches
- fitter une parabole
- en déduire d²E/dk² puis m*


In [ ]:
# TODO: Extraire un petit voisinage de k autour de VBM/CBM sur un segment choisi
# puis fitter E(k) = a k^2 + b k + c (ou en k-k0)
#
# Exemple de squelette:

def parabola(x, a, b, c):
    return a*x**2 + b*x + c

# x_data = ...  # vecteur k (en Å^-1 idéalement) ou paramètre de chemin
# E_data = ...  # énergie (eV)
# popt, pcov = curve_fit(parabola, x_data, E_data)
# a, b, c = popt
# d2E_dk2 = 2*a   # si x est k
# m_eff = (hbar**2) / (d2E_dk2 * 1.602176634e-19)  # conversion eV -> J si nécessaire
# print("m* =", m_eff, "kg ; en unités de me:", m_eff/9.10938356e-31)


# 4. Phonons — Bandes & vitesses du son


## 4.1 Télécharger la structure de bandes de phonons
- récupérer phonon band structure depuis MP (si disponible)
- tracer


In [ ]:
with MPRester(API_KEY) as mpr:
    ph_bs = mpr.get_phonon_bandstructure_by_material_id(MP_ID)

ph_plotter = PhononBSPlotter(ph_bs)
ph_plotter.get_plot()
plt.show()


## 4.2 Calculer la vitesse du son (3 branches acoustiques, 3 directions différentes)
Consigne :
- choisir **3 directions** différentes (segments proches de Γ)
- pour **3 branches acoustiques** différentes (LA, TA1, TA2), calculer :
v = dω/dq près de Γ

À faire :
- identifier les branches acoustiques (ω → 0 à Γ)
- pour chaque direction, approximer la pente sur les premiers points
- convertir en m/s (attention unités : ω souvent en THz ou rad/s selon source)


In [ ]:
# TODO: Extraction des fréquences et q près de Gamma.
# La structure de données dépend de ph_bs (pymatgen phonon band structure).
# Piste: ph_bs.bands, ph_bs.qpoints, ph_bs.branches

print("Nombre de branches (segments):", len(ph_bs.branches))
ph_bs.branches[:3]


In [ ]:
# TODO: Implémenter une estimation de pente:
# 1) sélectionner une branche proche de Γ (par ex. première branche)
# 2) prendre les 2-4 premiers points après Γ
# 3) pente = Δω/Δq
#
# IMPORTANT: vérifier unités
#
# Exemple de squelette:
# q = np.array([...])          # en 1/Å ou 1/m
# omega = np.array([...])      # en THz ou rad/s
# v = np.gradient(omega, q)[0] # pente près de Γ
# print(v)


# 5. DOS phonons & chaleur spécifique — Debye / Einstein


## 5.1 Télécharger la densité d'états phonons (DOS)
- tracer DOS(ω)
- préparer données pour Cv(T) fourni par l'objet MP (si disponible)


In [ ]:
with MPRester(API_KEY) as mpr:
    ph_dos = mpr.get_phonon_dos_by_material_id(MP_ID)

plt.figure()
plt.plot(ph_dos.frequencies, ph_dos.densities)
plt.xlabel("Fréquence")
plt.ylabel("DOS")
plt.title("Phonon DOS")
plt.show()


## 5.2 Chaleur spécifique du matériau (référence)
Selon les objets MP, vous pouvez avoir directement Cv(T).  
Sinon, vous pouvez la reconstruire à partir de la DOS phonons (plus long).

À remplir :
- méthode choisie (donnée MP / reconstruction)


In [ ]:
# TODO: Selon version MP/pymatgen, Cv(T) peut être accessible via ph_dos (ou via un autre endpoint).
# Afficher les attributs utiles:
dir(ph_dos)[:50]


## 5.3 Modèle d'Einstein : fit de ΘE (moindres carrés)
Formule (par mole) typique :
Cv_E(T) = 3 R (ΘE/T)^2 * exp(ΘE/T) / (exp(ΘE/T)-1)^2

À adapter selon l'échelle de vos données (par atome / par maille / par mole).


In [ ]:
R = 8.314462618  # J/mol/K

def Cv_einstein(T, theta_E):
    x = theta_E / T
    ex = np.exp(x)
    return 3 * R * (x**2) * ex / (ex - 1)**2

# TODO: T_data, Cv_data à définir à partir des données de référence
# popt, _ = curve_fit(Cv_einstein, T_data, Cv_data, p0=[300])
# theta_E = popt[0]
# theta_E


## 5.4 Modèle de Debye : fit de ΘD (moindres carrés)
Formule :
Cv_D(T) = 9 R (T/ΘD)^3 ∫_0^{ΘD/T} x^4 e^x/(e^x-1)^2 dx

On calcule numériquement l'intégrale.


In [ ]:
from scipy.integrate import quad

def Cv_debye(T, theta_D):
    def integrand(x):
        ex = np.exp(x)
        return (x**4) * ex / (ex - 1)**2
    x_max = theta_D / T
    integral, _ = quad(integrand, 0, x_max, limit=200)
    return 9 * R * (T/theta_D)**3 * integral

# TODO: fit
# popt, _ = curve_fit(Cv_debye, T_data, Cv_data, p0=[300])
# theta_D = popt[0]
# theta_D


## 5.5 Comparer les courbes de chaleur spécifique
- courbe référence (objet téléchargé / données MP)
- modèle Debye (ΘD optimal)
- modèle Einstein (ΘE optimal)


In [ ]:
# TODO: Remplacer par vos données
# plt.figure()
# plt.plot(T_data, Cv_data, label="Référence")
# plt.plot(T_data, Cv_debye(T_data, theta_D), label=f"Debye (ΘD={theta_D:.1f} K)")
# plt.plot(T_data, Cv_einstein(T_data, theta_E), label=f"Einstein (ΘE={theta_E:.1f} K)")
# plt.xlabel("T (K)")
# plt.ylabel("Cv")
# plt.legend()
# plt.show()


## 5.6 Représenter les trois densités d'états phonons correspondantes
Consigne :
- DOS "référence" (objet téléchargé)
- DOS modèle Debye
- DOS modèle Einstein

Remarque :
- Einstein : pics delta (ou gaussien étroit autour ωE)
- Debye : DOS ~ ω^2 jusqu'à ωD
Il faudra **normaliser** pour comparer.


In [ ]:
# TODO: Construire une grille de fréquences et tracer les DOS modèles (schéma).
# Exemple de squelette:

# w = np.linspace(0, w_max, 1000)
# dos_ref = ...
# dos_debye = ...
# dos_einstein = ...
# plt.plot(w, dos_ref, label="Référence")
# plt.plot(w, dos_debye, label="Debye")
# plt.plot(w, dos_einstein, label="Einstein")
# plt.legend()
# plt.show()


# 6. Conclusion (physique + lien avec concepts du cours)
À couvrir :
- Structure cristalline (maille, symétries, BZ)
- Diffraction (pics, hkl, cohérence avec structure)
- Électronique (gap, dispersion, masses effectives)
- Phonons (vitesses du son, branches)
- Thermique (ΘD, ΘE, qualité des fits, limites des modèles)
